In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('AEP_hourly.csv')

df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.set_index('Datetime')

def time(df):
    df['Month'] = df.index.month
    df['Year'] = df.index.year
    df['Dayofweek'] = df.index.dayofweek
    df['Dayofyear'] = df.index.dayofyear
    return df

def features(df):
    df['Lag1'] = df['AEP_MW'].shift(1)
    df['Lag2'] = df['AEP_MW'].shift(2)
    df['MA 24'] = df['AEP_MW'].shift(1).rolling(24).mean()
    df['MA 7'] = df['AEP_MW'].shift(1).rolling(7).mean()
    return df

q1 = df['AEP_MW'].quantile(0.25)
q3 = df['AEP_MW'].quantile(0.75)
iqr = q3 - q1

upper = q3 +1.5 * iqr
lower = q1 - 1.5 *iqr

df = df[(df['AEP_MW'] >= 11000) & (df['AEP_MW'] <= 20000)]

df = features(df)
df = time(df)

df.dropna(inplace = True)

def split(df,perc):
    split = int(len(df) * perc)
    train,test = df[0:split], df[split:]
    return train,test

train = df[(df.index >= '2004-12-29 01:00:00') & (df.index < '2015-03-10 04:00:00')]
test = df[(df.index >= '2015-03-09 05:00:00') & (df.index < '2018-01-03 00:00:00')]



Features = ['Lag1', 'Lag2', 'MA 24','MA 7', 'Month', 'Year', 'Dayofweek','Dayofyear']
Target = ['AEP_MW']

x_train = train[Features]
y_train = train[Target]

x_test = test[Features]
y_test = test[Target]
y_test = y_test.iloc[:,0]
model = XGBRegressor(n_estimators = 1500, learning_rate = 0.001, max_depth = 5)
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_test = model.predict(x_test)

pred_test = pd.Series(pred_test, index = y_test.index)
pred_train = pd.Series(pred_train, index = y_train.index)

mae_train = mean_absolute_error(y_train, pred_train)
print ("Train error:%.2f"%mae_train)
mae_test = mean_absolute_error(y_test, pred_test)
print ("Test Error:%.2f"%mae_test)

train['Predictions'] = pred_train
train['Residuals'] = train['AEP_MW'] - train['Predictions']

test['Predictions'] = pred_test
test['Residuals'] = test['AEP_MW'] - test['Predictions']

Features_res = ['Lag1', 'Lag2', 'MA 24','MA 7', 'Month', 'Year', 'Dayofweek','Dayofyear', 'Predictions']
Target_res = ['Residuals']

x_train_res = train[Features_res]
y_train_res = train[Target_res]

x_test_res = test[Features_res]
y_test_res = test[Target_res]

model_res = XGBRegressor(n_estimators = 1500, learning_rate = 0.001, max_depth = 3)
model_res.fit(x_train_res, y_train_res)

predict_train = model_res.predict(x_train_res)
predict_test = model_res.predict(x_test_res)

predict_final_train = pred_train + predict_train
predict_final_test = pred_test + predict_test


mae_finaL_train = mean_absolute_error(y_train, predict_final_train)
print ("TRAIN FINAL ERROR:%.2f"%mae_finaL_train)
mae_finaL_test = mean_absolute_error(y_test, predict_final_test)
print ("TEST FINAL ERROR:%.2f"%mae_finaL_test)

Train error:534.32
Test Error:569.23
TRAIN FINAL ERROR:321.88
TEST FINAL ERROR:324.78
